# Deliverable 3 - Body and Cloth Depth Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukas-sek/Pose_Estimation/blob/main/depth_estimation_d3.ipynb)

This notebook extends the PyTorch baseline with:
- Weak supervision: ranking loss and normal consistency loss
- Geometry-aware losses: scale-invariant depth loss
- Body pose + depth multi-tasking with a joint heatmap head and soft-argmax

It also provides ablation runs and analysis prompts for the required comparisons.

In [ ]:
import os, sys, time, pickle, math, gc
import numpy as np
import cv2
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'timm version: {timm.__version__}')

In [ ]:
BASE_DIR = os.getcwd()

CONFIG = {
    'root'         : os.path.join(BASE_DIR, 'preprocessed_data'),
    'raw_subset'   : os.path.join(BASE_DIR, 'cloth3d++_subset'),
    'img_size'     : 256,
    'batch_size'   : 8,
    'lr'           : 1e-4,
    'weight_decay' : 1e-4,
    'epochs'       : 30,
    'patience'     : 5,
    'encoder'      : 'deit',
    'decoder'      : 'bilinear',
    'patch_size'   : 16,
    'pretrained'   : True,
    'transfer'     : 'full_finetune',
    'checkpoint'   : os.path.join(BASE_DIR, 'best_model_d3.pth'),
    'history_path' : os.path.join(BASE_DIR, 'history_d3.pkl'),
    'joint_cache'  : os.path.join(BASE_DIR, 'preprocessed_data', 'joints_2d'),
    'rank_pairs'   : 2000,
    'rank_margin'  : 0.05,
    'si_lambda'    : 0.5,
}

print('Root:', CONFIG['root'])
print('Raw subset:', CONFIG['raw_subset'])

In [ ]:
def read_list(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

root = CONFIG['root']
train_list = read_list(os.path.join(root, 'train.txt'))
val_list   = read_list(os.path.join(root, 'validation.txt'))
test_list  = read_list(os.path.join(root, 'test.txt'))

print(f'Train: {len(train_list)}  Val: {len(val_list)}  Test: {len(test_list)}')

## Joint Projection Cache (SMPL -> 2D)

We project 14 SMPL joints to 2D, then apply the same crop + resize used in preprocessing.
Cached outputs are stored as .npz files with keys `joints` (14x2) and `vis` (14).

In [ ]:
# Add DataReader to path
DATAREADER_DIR = os.path.join(BASE_DIR, 'cloth3d', 'DataReader')
sys.path.insert(0, os.path.join(BASE_DIR, 'cloth3d'))
sys.path.insert(0, DATAREADER_DIR)

from DataReader.read import DataReader
from DataReader.util import proj, zRotMatrix

# 14-joint subset (SMPL indices):
# L/R hip, L/R knee, L/R ankle, neck, head, L/R shoulder, L/R elbow, L/R wrist
JOINT_IDXS = [1, 2, 4, 5, 7, 8, 12, 15, 16, 17, 18, 19, 20, 21]

def get_square_crop(mask, margin=10):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    cy = int((rmin + rmax) / 2)
    cx = int((cmin + cmax) / 2)
    half = max(rmax - rmin, cmax - cmin) // 2 + margin
    return cx, cy, half

def crop_bounds(mask, margin=10):
    H, W = mask.shape[:2]
    cx, cy, half = get_square_crop(mask, margin)
    x1 = max(0, cx - half)
    x2 = min(W, cx + half)
    y1 = max(0, cy - half)
    y2 = min(H, cy + half)
    return x1, y1, x2, y2

def project_points(P, pts3d):
    ones = np.ones((pts3d.shape[0], 1), dtype=np.float32)
    pts_h = np.concatenate([pts3d, ones], axis=1)
    proj_h = (P @ pts_h.T).T  # (N, 3)
    w = proj_h[:, 2:3]
    xy = proj_h[:, :2] / (w + 1e-9)
    z = w.squeeze(1)
    return xy, z

def build_crop_table(sample_dir, sample, needed_frames):
    segm_path = os.path.join(sample_dir, f'{sample}_segm.mkv')
    cap = cv2.VideoCapture(segm_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open segmentation video: {segm_path}')

    crop_table = {}
    frame_idx = 0
    needed = set(needed_frames)
    while True:
        ok, seg = cap.read()
        if not ok:
            break
        if frame_idx in needed:
            mask = seg[:, :, 0] > 0
            if mask.any():
                crop_table[frame_idx] = crop_bounds(mask, margin=10)
        frame_idx += 1
    cap.release()
    return crop_table

def build_joint_cache(sample_names, cfg):
    os.makedirs(cfg['joint_cache'], exist_ok=True)
    reader = DataReader()
    reader.SRC = cfg['raw_subset'] + os.sep

    # Map sample -> set(frames)
    sample_map = {}
    for name in sample_names:
        sample, frame = name.split('_')
        frame_idx = int(frame)
        sample_map.setdefault(sample, set()).add(frame_idx)

    total = len(sample_names)
    done = 0

    for sample, frames in sorted(sample_map.items()):
        sample_dir = os.path.join(cfg['raw_subset'], sample)
        info = reader.read_info(sample)
        P = proj(info['camLoc'])
        zrot = zRotMatrix(info['zrot'])

        gender = 'm' if info['gender'] else 'f'
        smpl = reader.smpl[gender]

        crop_table = build_crop_table(sample_dir, sample, frames)

        for frame_idx in sorted(frames):
            name = f'{sample}_{frame_idx}'
            out_path = os.path.join(cfg['joint_cache'], name + '.npz')
            if os.path.isfile(out_path):
                done += 1
                continue
            if frame_idx not in crop_table:
                continue

            pose = info['poses'][:, frame_idx].reshape(smpl.pose_shape)
            shape = info['shape']
            trans = info['trans'][:, frame_idx].reshape(smpl.trans_shape)

            _, J = smpl.set_params(pose=pose, beta=shape, trans=trans)
            J = J - J[0:1]
            J = zrot.dot(J.T).T
            J = J[JOINT_IDXS]

            xy, z = project_points(P, J)

            x1, y1, x2, y2 = crop_table[frame_idx]
            crop_w = max(1, x2 - x1)
            crop_h = max(1, y2 - y1)

            xy[:, 0] = (xy[:, 0] - x1) * (cfg['img_size'] / crop_w)
            xy[:, 1] = (xy[:, 1] - y1) * (cfg['img_size'] / crop_h)

            vis = (z > 0) & (xy[:, 0] >= 0) & (xy[:, 0] < cfg['img_size']) & \n
                  (xy[:, 1] >= 0) & (xy[:, 1] < cfg['img_size'])

            np.savez(out_path, joints=xy.astype(np.float32), vis=vis.astype(np.uint8))
            done += 1

        if done % 500 == 0 or done == total:
            print(f'Cached {done}/{total} joint files')

    print('Joint cache ready:', cfg['joint_cache'])

# Build cache once (skip if files already exist)
all_names = train_list + val_list + test_list
build_joint_cache(all_names, CONFIG)

## Dataset
RGB + depth + 2D joints (cached) + visibility mask.

In [ ]:
class Cloth3DDatasetD3(Dataset):
    def __init__(self, data_list, root, joint_cache, img_size=256, augment=False):
        self.data_list = data_list
        self.root = root
        self.joint_cache = joint_cache
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        name = self.data_list[idx]

        dpt = np.load(os.path.join(self.root, 'depth', name + '.npy')).astype(np.float32)
        mask = dpt > 0
        if mask.any():
            dpt[mask] = (dpt[mask] - dpt[mask].min() + 0.001) / 2.0

        img = cv2.imread(os.path.join(self.root, 'image', name + '.jpg'))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)

        s = self.img_size
        if img.shape[:2] != (s, s):
            img = cv2.resize(img, (s, s))
            dpt = cv2.resize(dpt, (s, s), interpolation=cv2.INTER_NEAREST)
            mask = dpt > 0

        if mask.any():
            for c in range(3):
                ch = img[:, :, c]
                mu, sigma = ch[mask].mean(), ch[mask].std()
                img[:, :, c] = (ch - mu) / (sigma + 1e-5)

        if self.augment:
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :].copy()
                dpt = dpt[:, ::-1].copy()
            tx, ty = np.random.randint(-20, 20, 2).tolist()
            M = np.float32([[1, 0, tx], [0, 1, ty]])
            img = cv2.warpAffine(img, M, (s, s))
            dpt = cv2.warpAffine(dpt, M, (s, s))

        joints = np.zeros((len(JOINT_IDXS), 2), dtype=np.float32)
        vis = np.zeros((len(JOINT_IDXS),), dtype=np.uint8)
        jp = os.path.join(self.joint_cache, name + '.npz')
        if os.path.isfile(jp):
            data = np.load(jp)
            joints = data['joints'].astype(np.float32)
            vis = data['vis'].astype(np.uint8)

        img = torch.from_numpy(img).permute(2, 0, 1)
        dpt = torch.from_numpy(dpt).unsqueeze(0)
        joints = torch.from_numpy(joints)
        vis = torch.from_numpy(vis)
        return img, dpt, joints, vis

train_ds = Cloth3DDatasetD3(train_list, CONFIG['root'], CONFIG['joint_cache'], CONFIG['img_size'], augment=True)
val_ds   = Cloth3DDatasetD3(val_list,   CONFIG['root'], CONFIG['joint_cache'], CONFIG['img_size'], augment=False)
test_ds  = Cloth3DDatasetD3(test_list,  CONFIG['root'], CONFIG['joint_cache'], CONFIG['img_size'], augment=False)

## Model (Depth + Joint Heatmaps)
Multi-task decoder with a depth head and a 14-channel joint heatmap head.

In [ ]:
class DeiTEncoder(nn.Module):
    def __init__(self, patch_size=16, pretrained=True, img_size=256):
        super().__init__()
        use_pretrained = pretrained and (patch_size == 16)
        kwargs = dict(pretrained=use_pretrained, img_size=img_size, num_classes=0)
        if patch_size != 16:
            kwargs['patch_size'] = patch_size
        self.backbone = timm.create_model('deit_small_patch16_224', **kwargs)
        self.embed_dim = self.backbone.embed_dim
        self.out_channels = [self.embed_dim]

    def forward(self, x):
        B = x.shape[0]
        feats = self.backbone.forward_features(x)
        patches = feats[:, 1:]
        N = patches.shape[1]
        H = W = int(N ** 0.5)
        spatial = patches.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        return [spatial]

class SwinEncoder(nn.Module):
    def __init__(self, pretrained=True, img_size=256):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=img_size,
        )
        self.out_channels = self.backbone.feature_info.channels()

    def forward(self, x):
        return self.backbone(x)

class EfficientFormerEncoder(nn.Module):
    def __init__(self, pretrained=True, img_size=256):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientformer_l1',
            pretrained=pretrained,
            features_only=True,
        )
        self.out_channels = self.backbone.feature_info.channels()

    def forward(self, x):
        return self.backbone(x)

class MultiBilinearDecoder(nn.Module):
    def __init__(self, in_channels_list, target_size=256, num_joints=14):
        super().__init__()
        self.target_size = target_size
        in_ch = in_channels_list[-1]
        self.trunk = nn.Sequential(
            nn.Conv2d(in_ch, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.depth_head = nn.Sequential(nn.Conv2d(64, 1, 1), nn.Sigmoid())
        self.joint_head = nn.Conv2d(64, num_joints, 1)

    def forward(self, features):
        x = features[-1]
        x = F.interpolate(x, size=(self.target_size, self.target_size), mode='bilinear', align_corners=False)
        x = self.trunk(x)
        depth = self.depth_head(x)
        joints = self.joint_head(x)
        return depth, joints

class MultiFPNDecoder(nn.Module):
    def __init__(self, in_channels_list, target_size=256, fpn_channels=128, num_joints=14):
        super().__init__()
        self.target_size = target_size
        self.laterals = nn.ModuleList([nn.Conv2d(c, fpn_channels, 1) for c in in_channels_list])
        self.out_convs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1), nn.ReLU(inplace=True))
            for _ in in_channels_list
        ])
        n = len(in_channels_list)
        self.head = nn.Sequential(
            nn.Conv2d(fpn_channels * n, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.depth_head = nn.Sequential(nn.Conv2d(64, 1, 1), nn.Sigmoid())
        self.joint_head = nn.Conv2d(64, num_joints, 1)

    def forward(self, features):
        lats = [l(f) for l, f in zip(self.laterals, features)]
        for i in range(len(lats) - 1, 0, -1):
            lats[i - 1] = lats[i - 1] + F.interpolate(
                lats[i], size=lats[i - 1].shape[-2:], mode='bilinear', align_corners=False
            )
        ups = [
            F.interpolate(conv(lat), size=(self.target_size, self.target_size), mode='bilinear', align_corners=False)
            for conv, lat in zip(self.out_convs, lats)
        ]
        x = self.head(torch.cat(ups, dim=1))
        depth = self.depth_head(x)
        joints = self.joint_head(x)
        return depth, joints

class MultiTaskModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        return self.decoder(self.encoder(x))

def build_model(cfg):
    enc_name = cfg['encoder']
    img_size = cfg['img_size']
    pretrained = cfg['pretrained']

    if enc_name == 'deit':
        encoder = DeiTEncoder(patch_size=cfg['patch_size'], pretrained=pretrained, img_size=img_size)
    elif enc_name == 'swin':
        encoder = SwinEncoder(pretrained=pretrained, img_size=img_size)
    elif enc_name == 'efficientformer':
        encoder = EfficientFormerEncoder(pretrained=pretrained, img_size=img_size)
    else:
        raise ValueError(f'Unknown encoder: {enc_name}')

    if cfg['decoder'] == 'bilinear':
        decoder = MultiBilinearDecoder(encoder.out_channels, target_size=img_size, num_joints=len(JOINT_IDXS))
    elif cfg['decoder'] == 'fpn':
        decoder = MultiFPNDecoder(encoder.out_channels, target_size=img_size, num_joints=len(JOINT_IDXS))
    else:
        raise ValueError(f'Unknown decoder: {cfg[
]}')

    return MultiTaskModel(encoder, decoder)

def apply_transfer_learning(model, mode):
    if mode == 'full_finetune':
        for p in model.parameters():
            p.requires_grad = True
    elif mode == 'frozen_backbone':
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.decoder.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f'Unknown transfer mode: {mode}')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Transfer learning: {mode}')
    print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

## Losses and Metrics
Includes ranking loss, normal consistency, and scale-invariant depth loss.

In [ ]:
def rmse(pred, gt):
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    return torch.sqrt(F.mse_loss(pred[mask], gt[mask]))

def depth_to_normals(depth, eps=1e-6):
    dz_dx = depth[:, :, :, 2:] - depth[:, :, :, :-2]
    dz_dy = depth[:, :, 2:, :] - depth[:, :, :-2, :]
    dz_dx = F.pad(dz_dx, [1, 1, 0, 0])
    dz_dy = F.pad(dz_dy, [0, 0, 1, 1])
    ones = torch.ones_like(dz_dx)
    n = torch.cat([-dz_dx, -dz_dy, ones], dim=1)
    return n / (torch.norm(n, dim=1, keepdim=True) + eps)

def mean_angular_error(pred_depth, gt_depth):
    mask = (gt_depth > 0).squeeze(1)
    n_pred = depth_to_normals(pred_depth)
    n_gt = depth_to_normals(gt_depth)
    cos_sim = (n_pred * n_gt).sum(dim=1).clamp(-1.0, 1.0)
    angle = torch.acos(cos_sim) * (180.0 / torch.pi)
    if not mask.any():
        return torch.tensor(0.0, device=pred_depth.device)
    return angle[mask].mean()

def depth_l1_loss(pred, gt):
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    return F.l1_loss(pred[mask], gt[mask])

def normal_consistency_loss(pred, gt):
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    n_pred = depth_to_normals(pred)
    n_gt = depth_to_normals(gt)
    return F.l1_loss(n_pred[mask.squeeze(1)], n_gt[mask.squeeze(1)])

def scale_invariant_loss(pred, gt, lam=0.5):
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    pred = pred[mask]
    gt = gt[mask]
    d = torch.log(pred + 1e-6) - torch.log(gt + 1e-6)
    n = d.numel()
    return (d.pow(2).sum() / n) - (lam * (d.sum().pow(2)) / (n * n))

def ranking_loss(pred, gt, n_pairs=2000, margin=0.05):
    mask = (gt > 0).squeeze(1)
    if mask.sum() < 2:
        return torch.tensor(0.0, device=pred.device)

    pred_flat = pred.squeeze(1).reshape(pred.shape[0], -1)
    gt_flat = gt.squeeze(1).reshape(gt.shape[0], -1)
    mask_flat = mask.reshape(mask.shape[0], -1)

    losses = []
    for b in range(pred.shape[0]):
        valid_idx = torch.where(mask_flat[b])[0]
        if valid_idx.numel() < 2:
            continue
        idx = valid_idx[torch.randint(0, valid_idx.numel(), (n_pairs * 2,), device=pred.device)]
        i1, i2 = idx[:n_pairs], idx[n_pairs:]
        d1 = gt_flat[b, i1]
        d2 = gt_flat[b, i2]
        s = torch.sign(d1 - d2)
        s[s == 0] = 1
        p1 = pred_flat[b, i1]
        p2 = pred_flat[b, i2]
        losses.append(F.relu(margin - s * (p1 - p2)).mean())
    if not losses:
        return torch.tensor(0.0, device=pred.device)
    return torch.stack(losses).mean()

def soft_argmax_2d(heatmaps, beta=100.0):
    B, J, H, W = heatmaps.shape
    heat = heatmaps.reshape(B, J, -1)
    probs = F.softmax(heat * beta, dim=-1)
    ys = torch.linspace(0, H - 1, H, device=heatmaps.device)
    xs = torch.linspace(0, W - 1, W, device=heatmaps.device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
    grid = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=0)
    grid = grid.unsqueeze(0).unsqueeze(0)
    coords = torch.sum(probs.unsqueeze(2) * grid, dim=-1)
    return coords  # (B, J, 2)

def joint_l1_loss(pred_xy, gt_xy, vis):
    vis = vis.float().unsqueeze(-1)
    if vis.sum() < 1:
        return torch.tensor(0.0, device=pred_xy.device)
    return (torch.abs(pred_xy - gt_xy) * vis).sum() / vis.sum()

## Training and Ablations
Configure three regimes: full supervision, weak-only, geometry-aware.

In [ ]:
LOSS_CONFIGS = {
    'full': {
        'depth_l1' : 1.0,
        'joint_l1' : 1.0,
        'normal'   : 0.5,
        'scale_inv': 0.5,
        'rank'     : 0.2,
    },
    'weak': {
        'depth_l1' : 0.0,
        'joint_l1' : 0.0,
        'normal'   : 1.0,
        'scale_inv': 0.0,
        'rank'     : 1.0,
    },
    'geom': {
        'depth_l1' : 1.0,
        'joint_l1' : 1.0,
        'normal'   : 0.0,
        'scale_inv': 1.0,
        'rank'     : 0.0,
    },
}

def train_epoch(model, loader, optimizer, device, loss_cfg):
    model.train()
    totals = {k: 0.0 for k in ['total', 'depth_l1', 'joint_l1', 'normal', 'scale_inv', 'rank']}
    for X, Y, J, V in loader:
        X, Y, J, V = X.to(device), Y.to(device), J.to(device), V.to(device)
        pred_depth, pred_heat = model(X)

        pred_xy = soft_argmax_2d(pred_heat)

        l_depth = depth_l1_loss(pred_depth, Y)
        l_joint = joint_l1_loss(pred_xy, J, V)
        l_norm  = normal_consistency_loss(pred_depth, Y)
        l_si    = scale_invariant_loss(pred_depth, Y, lam=CONFIG['si_lambda'])
        l_rank  = ranking_loss(pred_depth, Y, n_pairs=CONFIG['rank_pairs'], margin=CONFIG['rank_margin'])

        total = (loss_cfg['depth_l1'] * l_depth +
                 loss_cfg['joint_l1'] * l_joint +
                 loss_cfg['normal']   * l_norm +
                 loss_cfg['scale_inv'] * l_si +
                 loss_cfg['rank']     * l_rank)

        optimizer.zero_grad()
        total.backward()
        optimizer.step()

        totals['total'] += total.item()
        totals['depth_l1'] += l_depth.item()
        totals['joint_l1'] += l_joint.item()
        totals['normal'] += l_norm.item()
        totals['scale_inv'] += l_si.item()
        totals['rank'] += l_rank.item()

    n = max(1, len(loader))
    return {k: v / n for k, v in totals.items()}

@torch.no_grad()
def val_epoch(model, loader, device):
    model.eval()
    total_rmse, total_mae_n, total_joint = 0.0, 0.0, 0.0
    n = 0
    for X, Y, J, V in loader:
        X, Y, J, V = X.to(device), Y.to(device), J.to(device), V.to(device)
        pred_depth, pred_heat = model(X)
        pred_xy = soft_argmax_2d(pred_heat)
        total_rmse += rmse(pred_depth, Y).item()
        total_mae_n += mean_angular_error(pred_depth, Y).item()
        total_joint += joint_l1_loss(pred_xy, J, V).item()
        n += 1
    return total_rmse / n, total_mae_n / n, total_joint / n

def train(model, cfg, train_list, val_list, loss_cfg, tag='run'):
    train_dl = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])

    best_rmse, patience_counter = float('inf'), 0
    history = {
        'train_total': [], 'train_depth_l1': [], 'train_joint_l1': [],
        'train_normal': [], 'train_scale_inv': [], 'train_rank': [],
        'val_rmse': [], 'val_mae_n': [], 'val_joint_l1': []
    }

    for epoch in range(cfg['epochs']):
        t0 = time.time()
        tr = train_epoch(model, train_dl, optimizer, DEVICE, loss_cfg)
        v_rmse, v_mae_n, v_joint = val_epoch(model, val_dl, DEVICE)
        scheduler.step()

        history['train_total'].append(tr['total'])
        history['train_depth_l1'].append(tr['depth_l1'])
        history['train_joint_l1'].append(tr['joint_l1'])
        history['train_normal'].append(tr['normal'])
        history['train_scale_inv'].append(tr['scale_inv'])
        history['train_rank'].append(tr['rank'])
        history['val_rmse'].append(v_rmse)
        history['val_mae_n'].append(v_mae_n)
        history['val_joint_l1'].append(v_joint)

        print(f'[{tag}] Epoch {epoch+1:3d}/{cfg[
]} | '
              f'loss={tr[
]:.4f} | val_RMSE={v_rmse:.4f} | '
              f'val_NormalMAE={v_mae_n:.2f} | val_JointL1={v_joint:.2f} | '
              f'{time.time()-t0:.1f}s')

        if v_rmse < best_rmse:
            best_rmse = v_rmse
            torch.save(model.state_dict(), cfg['checkpoint'])
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg['patience']:
                print(f'Early stopping at epoch {epoch+1} (best val RMSE {best_rmse:.4f})')
                break

    with open(cfg['history_path'], 'wb') as fh:
        pickle.dump(history, fh)
    print(f'Best val RMSE: {best_rmse:.4f}')
    return history

def run_experiment(tag, loss_key):
    cfg = {**CONFIG,
        'checkpoint': os.path.join(BASE_DIR, f'model_{tag}.pth'),
        'history_path': os.path.join(BASE_DIR, f'hist_{tag}.pkl'),
    }
    model = build_model(cfg).to(DEVICE)
    apply_transfer_learning(model, cfg['transfer'])
    return train(model, cfg, train_list, val_list, LOSS_CONFIGS[loss_key], tag=tag)

### Run the three regimes (uncomment to execute)
- full: all losses
- weak: ranking + normal consistency
- geom: scale-invariant + joint and depth supervision

In [ ]:
# h_full = run_experiment('full', 'full')
# h_weak = run_experiment('weak', 'weak')
# h_geom = run_experiment('geom', 'geom')

## Analysis Prompts
Use the results and visualizations below to answer:
- Full supervision vs weak-only vs geometry-aware
- Failure modes: textured vs smooth areas, around edges, etc.
- Stability vs accuracy
- Losses: convergence speed, overfitting tendencies

### 1. Compare Convergence Curves (Full vs Weak vs Geom)

Load the three history files and plot loss/metric curves side by side.

In [ ]:
import os, pickle
import matplotlib.pyplot as plt
import numpy as np

tags = ['full', 'weak', 'geom']
histories = {}

for tag in tags:
    hist_path = os.path.join(BASE_DIR, f'hist_{tag}.pkl')
    if os.path.isfile(hist_path):
        with open(hist_path, 'rb') as f:
            histories[tag] = pickle.load(f)
        print(f'Loaded {tag}: {len(histories[tag]["val_rmse"])} epochs')
    else:
        print(f'Missing {hist_path}')

if len(histories) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Convergence Comparison: Full vs Weak vs Geom', fontsize=14, fontweight='bold')

    metrics = [
        ('train_total', 'Train Loss', 0, 0),
        ('val_rmse', 'Val RMSE', 0, 1),
        ('val_mae_n', 'Val Normal MAE (°)', 0, 2),
        ('val_joint_l1', 'Val Joint L1', 1, 0),
        ('train_normal', 'Train Normal Loss', 1, 1),
        ('train_rank', 'Train Ranking Loss', 1, 2),
    ]

    for key, title, i, j in metrics:
        for tag, hist in histories.items():
            if key in hist:
                axes[i, j].plot(hist[key], label=tag, linewidth=2)
        axes[i, j].set_title(title)
        axes[i, j].set_xlabel('Epoch')
        axes[i, j].legend()
        axes[i, j].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'convergence_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to convergence_comparison.png')

### 2. Convergence Speed & Overfitting Analysis

Compute epochs to reach target RMSE and analyze train/val divergence.

In [ ]:
print('\n' + '='*70)
print('CONVERGENCE SPEED & OVERFITTING ANALYSIS')
print('='*70)

target_rmse_levels = [0.15, 0.10, 0.08]

for tag, hist in histories.items():
    print(f'\n{tag.upper()} regime:')
    print(f'  Final val RMSE: {hist["val_rmse"][-1]:.4f}')
    print(f'  Final val Normal MAE: {hist["val_mae_n"][-1]:.2f}°')
    print(f'  Final train loss: {hist["train_total"][-1]:.4f}')

    for target in target_rmse_levels:
        epochs_to_target = None
        for ep, rmse in enumerate(hist['val_rmse']):
            if rmse <= target:
                epochs_to_target = ep + 1
                break
        if epochs_to_target:
            print(f'  Epochs to reach RMSE {target:.2f}: {epochs_to_target}')
        else:
            print(f'  Did not reach RMSE {target:.2f}')

    # Overfitting: train loss vs val RMSE
    final_train = hist['train_total'][-1]
    final_val = hist['val_rmse'][-1]
    overfitting = 100 * (final_val - final_train) / final_train if final_train > 0 else 0
    print(f'  Overfitting indicator (val/train divergence): {overfitting:.1f}%')

print('\n' + '='*70)

### 3. Stability vs Accuracy Trade-off

Compute variance of metrics and stability scores across epochs.

In [ ]:
print('\n' + '='*70)
print('STABILITY vs ACCURACY ANALYSIS')
print('='*70)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Stability vs Accuracy: Variance of Metrics', fontsize=14, fontweight='bold')

stability_data = {}

for tag, hist in histories.items():
    print(f'\n{tag.upper()} regime:')

    val_rmse = np.array(hist['val_rmse'])
    val_mae = np.array(hist['val_mae_n'])

    rmse_mean = val_rmse.mean()
    rmse_std = val_rmse.std()
    mae_mean = val_mae.mean()
    mae_std = val_mae.std()

    print(f'  Val RMSE: mean={rmse_mean:.4f}, std={rmse_std:.4f} (lower std = more stable)')
    print(f'  Val Normal MAE: mean={mae_mean:.2f}°, std={mae_std:.2f}° (lower std = more stable)')

    # Stability score: inverse of coefficient of variation
    rmse_cv = rmse_std / rmse_mean if rmse_mean > 0 else float('inf')
    mae_cv = mae_std / mae_mean if mae_mean > 0 else float('inf')
    stability_score = 1.0 / (rmse_cv + mae_cv + 1e-6)
    print(f'  Stability score (higher = more stable): {stability_score:.4f}')

    stability_data[tag] = {
        'rmse_mean': rmse_mean,
        'rmse_std': rmse_std,
        'mae_mean': mae_mean,
        'mae_std': mae_std,
        'stability': stability_score,
    }

# Plots
tags_list = list(stability_data.keys())
rmse_means = [stability_data[t]['rmse_mean'] for t in tags_list]
rmse_stds = [stability_data[t]['rmse_std'] for t in tags_list]
mae_means = [stability_data[t]['mae_mean'] for t in tags_list]
mae_stds = [stability_data[t]['mae_std'] for t in tags_list]
stabilities = [stability_data[t]['stability'] for t in tags_list]

x = np.arange(len(tags_list))
width = 0.35

axes[0].bar(x - width/2, rmse_means, width, label='Mean', color='skyblue')
axes[0].bar(x + width/2, rmse_stds, width, label='Std', color='coral')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Val RMSE: Mean & Std')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tags_list)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x - width/2, mae_means, width, label='Mean', color='skyblue')
axes[1].bar(x + width/2, mae_stds, width, label='Std', color='coral')
axes[1].set_ylabel('Normal MAE (°)')
axes[1].set_title('Val Normal MAE: Mean & Std')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tags_list)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(tags_list, stabilities, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[2].set_ylabel('Stability Score')
axes[2].set_title('Overall Stability (higher = better)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'stability_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved to stability_analysis.png')
print('='*70)

### 4. Summary Table: Full vs Weak vs Geometry-Aware

Compare all three regimes with a summary table and interpretation guide.

In [ ]:
import pandas as pd

print('\n' + '='*90)
print('SUMMARY: Full Supervision vs Weak-Only vs Geometry-Aware')
print('='*90)

summary_rows = []
if 'stability_data' in dir():
    for tag, hist in histories.items():
        summary_rows.append({
            'Regime': tag.upper(),
            'Final RMSE': f"{hist['val_rmse'][-1]:.4f}",
            'Final Normal MAE (°)': f"{hist['val_mae_n'][-1]:.2f}",
            'Final Joint L1': f"{hist['val_joint_l1'][-1]:.4f}",
            'Best RMSE': f"{min(hist['val_rmse']):.4f}",
            'Train Loss': f"{hist['train_total'][-1]:.4f}",
            'Stability': f"{stability_data[tag]['stability']:.4f}",
        })

    df = pd.DataFrame(summary_rows)
    print('\n', df.to_string(index=False))
else:
    print('Run the previous cells first to compute stability_data.')

print('\n' + '='*90)
print('INTERPRETATION GUIDE:')
print('='*90)
print('''
• FULL regime: All losses combined (depth, joints, normals, scale-invariant, ranking).
  Expected: Best overall accuracy, balanced performance across metrics.

• WEAK regime: Only ranking + normal consistency (no direct depth/joint supervision).
  Expected: Potentially faster convergence but lower absolute accuracy; check overfitting.

• GEOM regime: Scale-invariant + depth + joint supervision.
  Expected: Better handling of depth scale ambiguity; smoother, more stable predictions.

Questions to address in your analysis:
  1. Which regime achieves the lowest RMSE and normal MAE?
  2. Does weak supervision converge faster but plateau earlier?
  3. Does geometry-aware loss produce more stable/smooth predictions?
  4. Are there speed/accuracy trade-offs? Which is most important for your application?
  5. How do failure modes differ: textured vs smooth areas, edge handling?
''')
print('='*90)